# Proving Speech-to-Text Accuracy

In order to justify our choice of STT (Speech-to-Text) technology, we need to prove its accuracy.
Word Error Rate (WER) is the standard metric for measuring the performance of an ASR (Automatic Speech Recognition) system.

$$ WER = \frac{Substitutions + Deletions + Insertions}{Total\ Words\ in\ Reference} $$

Since our browser extension runs in the user's browser, we'll simulate the accuracy testing here using standard Python metrics.

In [ ]:
def calculate_wer(reference, hypothesis):
    """
    Calculate the Word Error Rate (WER).
    reference: The ground truth string.
    hypothesis: The string output by our STT engine.
    """
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    
    # Create a matrix for the Levenshtein distance calculation
    d = [[0 for _ in range(len(hyp_words) + 1)] for _ in range(len(ref_words) + 1)]
    
    for i in range(len(ref_words) + 1):
        d[i][0] = i
    for j in range(len(hyp_words) + 1):
        d[0][j] = j
        
    for i in range(1, len(ref_words) + 1):
        for j in range(1, len(hyp_words) + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                d[i][j] = d[i - 1][j - 1]
            else:
                sub_cost = d[i - 1][j - 1] + 1
                del_cost = d[i - 1][j] + 1
                ins_cost = d[i][j - 1] + 1
                d[i][j] = min(sub_cost, del_cost, ins_cost)
                
    wer = float(d[len(ref_words)][len(hyp_words)]) / len(ref_words)
    return wer


### Simulating Test Cases
Let's assume we collected 3 samples of audio, transcribed them manually (Ground Truth), and then passed them through our Web Speech API (Hypothesis).

In [ ]:
test_cases = [
    {
        "ground_truth": "Hello world this is a test of the dictation extension",
        "hypothesis": "Hello world this is a test of dictation extension" # (1 deletion)
    },
    {
        "ground_truth": "The quick brown fox jumps over the lazy dog",
        "hypothesis": "The quick brown fox jumped over the lazy dogs" # (2 substitutions)
    },
    {
        "ground_truth": "Testing input fields on complex modern web applications",
        "hypothesis": "Testing input fields on complex modern web applications" # (Perfect)
    }
]

total_wer = 0
for i, tc in enumerate(test_cases):
    wer = calculate_wer(tc["ground_truth"], tc["hypothesis"])
    total_wer += wer
    print(f"Test Case {i+1}:")
    print(f"  Truth: {tc['ground_truth']}")
    print(f"  Output: {tc['hypothesis']}")
    print(f"  WER: {wer * 100:.2f}%\n")

average_wer = total_wer / len(test_cases)
print(f"Average Word Error Rate across all tests: {average_wer * 100:.2f}%")
print("Any score below 15% is generally considered highly acceptable for conversational dictation tools!")